In [19]:
import pandas as pd

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"
columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'tip_amount','total_amount']
df = pd.read_parquet(url, columns=columns)
# Fill NaN values for integer columns with 0
df['passenger_count'] = df['passenger_count'].fillna(0).astype(int)

# Optional: Also fill other numeric columns if they have NaNs
df['trip_distance'] = df['trip_distance'].fillna(0.0)
df['tip_amount'] = df['tip_amount'].fillna(0.0)
df['total_amount'] = df['total_amount'].fillna(0.0)
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1,4.07,6.82,34.12


In [20]:
from hw_models import Ride, ride_from_row, ride_serializer

ride = ride_from_row(df.iloc[0])
ride

Ride(lpep_pickup_datetime='2025-10-01 00:21:47', lpep_dropoff_datetime='2025-10-01 00:24:37', PULocationID=247, DOLocationID=69, passenger_count=1, trip_distance=0.7, tip_amount=1.7, total_amount=10.0)

In [21]:
from kafka import KafkaProducer

server = 'localhost:9092'
topic_name = 'green-trips'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [24]:
import time

t0 = time.time()
row_count = 0  # Initialize counter

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    row_count += 1  # Increment for every row
    # print(f"Sent: {ride}")
    # time.sleep(0.01)

producer.flush()

t1 = time.time()

print(f'Sent {row_count} rows')
print(f'took {(t1 - t0):.2f} seconds')

Sent 49416 rows
took 18.80 seconds
